In [4]:
import torch

print (torch.cuda.is_available())
print (torch.cuda.get_device_name(0))

!pip install -q transformers accelerate bitsandbytes
#transformers -> hugging face model/tokenizer library
#accelerate -> helps place models on gpu auto
#bitsandbytes -> enables 4 bit/8 bit loading



True
Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.7 MB/s eta 0:00:00


In [5]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [7]:
#full pipeline for below:

# model_id
#   ↓
# loads matching tokenizer
#   ↓
# loads matching model
#   ↓
# messages written as role/content dictionaries
#   ↓
# apply_chat_template formats messages for Llama
#   ↓
# tokenizer converts formatted prompt into token IDs
#   ↓
# model.generate creates prompt tokens + new answer tokens
#   ↓
# slice removes prompt tokens
#   ↓
# tokenizer.decode converts answer tokens back to English



model_id = "meta-llama/Llama-3.2-3B-Instruct" #hugging face model name. Llama-3.2-3B = model family and size. Instruct = chat/instruction tuned

#AutoTokenizer: Loads the correct tokenizer for the model.
#AutoModelForCausalLM: Loads a text-generation model.
# BitsAndBytesConfig: Stores the 4-bit quantization settings.

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig (
    load_in_4bit = True, #Load the model weights in 4-bit instead of fp16/fp32
    bnb_4bit_quant_type= "nf4",
    bnb_4bit_compute_dtype= torch.float16, #Store weights in 4-bit, but do some computation in float16
    bnb_4bit_use_double_quant=True,
)
#load tokenizer: converst text to ids ex. numbers
tokenizer =AutoTokenizer.from_pretrained (model_id)

#This loads the actual neural network, the “machine” that takes tokens and predicts output tokens.
model = AutoModelForCausalLM.from_pretrained(model_id, #load pretrained model from huggingface
                                            quantization_config=bnb_config, #Use our 4-bit settings.So instead of loading the model normally, it compresses the weights into 4-bit.
                                            device_map="auto") #auto decide where to put model parts. here on gpu
#create message
messages = [
    {"role": "system", "content": "You are a helpful AI tutor who explains clearly and briefly."},
    {"role": "user","content": "What is retrieval-augmented generation? Explain simply."}
    ]

# This does two jobs:
# 1. Formats the chat messages in the exact style Llama expects
# 2. Converts that formatted text into token IDs
inputs = tokenizer.apply_chat_template( #Take my clean messages list and convert it into the exact prompt format this model was trained to understand
    messages,
    add_generation_prompt=True, #This adds the signal that says: Now it is the assistant’s turn to answer.
    return_tensors="pt" #return pytorch tensors. tensor = data structure pytroch models use
).to(model.device) #Move the input tokens to the same device as the model. If model is on GPU, input should be on GPU too.

#generate output
output = model.generate (**inputs,  #inputs is no longer English text.A batch of token IDs ready for the model.
                         max_new_tokens=200, #generate at most 200 tokens. token doesnt equal word.
                         do_sample=True, #Do not always pick the most obvious next token, Allow some randomness.
                         temperature=0.7, #Temperature controls randomness, higher= more creative
                         top_p=0.9, #top_p limits the model to reasonable next-word options.
                         )
#outputs contains: original input tokens + newly generated tokens but we only need the output and not the input. so:
#inputs.shape[-1] gives the number of tokens in the input prompt. Example: inputs = [[10, 20, 30, 40]] Shape is: [1, 4] Meaning: 1 example, 4 tokens So: inputs.shape[-1] is: 4 That tells us: The prompt was 4 tokens long.
#What does outputs[0][inputs.shape[-1]:] mean? Using the example: outputs[0] = [10, 20, 30, 40, 50, 60, 70] inputs.shape[-1] = 4 Then: outputs[0][4:] means: Start at index 4 and take everything after it. Result: [50, 60, 70] That is only the newly generated answer tokens.
answer = tokenizer.decode (output[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)

print(answer)


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Retrieval-Augmented Generation (RAG) is a technique used in artificial intelligence (AI) and machine learning (ML) to improve the performance of language models.

In simple terms, RAG works by combining two models:

1. **Language Model (LM)**: generates text based on its understanding of language patterns.
2. **Knowledge Retrieval Model (KR)**: searches and retrieves relevant information from a knowledge base or database.

The idea is to ask the LM to generate text, but also to retrieve relevant information from the knowledge base to inform the generation process. This allows the LM to produce more accurate, informative, and context-specific text.

Think of it like a librarian: the KR model is like the librarian who retrieves books (information) on a specific topic, and the LM is like the writer who uses those books to generate a new text.


In [8]:
def ask(
    prompt,
    system="You are a concise technical tutor.",
    temperature=0.7,
    top_p=0.9,
    top_k=None,
    repetition_penalty=1.0,
    max_new_tokens=120,
    do_sample=True,
):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "repetition_penalty": repetition_penalty,
    }

    if do_sample:
        gen_kwargs["temperature"] = temperature
        gen_kwargs["top_p"] = top_p
        if top_k is not None:
            gen_kwargs["top_k"] = top_k

    outputs = model.generate(
        **inputs,
        **gen_kwargs
    )

    answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    return answer

In [11]:
#The 5 required prompting exercises

# Exercise 1 — Zero-shot
# Zero-shot means: # Ask directly with no examples.

print(ask(
    "Classify the sentiment of: 'The battery life is terrible.'",
    temperature=0.0,
    do_sample=False,
    max_new_tokens=50
))

#Exercise 3 — Chain-of-thought comparison
print (ask("If a shirt costs $20 and is discounted by 25%, what is the final price?",
    temperature=0.0,
    do_sample=False,
    max_new_tokens=80))


#excerise 4

code_to_review = """
def add_numbers(a, b):
    return str(a + b)
"""
print(ask(
    f"Review this Python code:\n{code_to_review}",
    system="You are a senior Python developer reviewing code for correctness, type safety, and maintainability.",
    temperature=0.3,
    top_p=0.9,
    max_new_tokens=150
))

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


The sentiment of the statement 'The battery life is terrible' is negative.


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


To find the final price, first calculate the discount amount:

Discount = 25% of $20
= 0.25 * $20
= $5

Now, subtract the discount from the original price:

Final Price = Original Price - Discount
= $20 - $5
= $15

The final price of the shirt is $15.
**Code Review**

The provided Python function `add_numbers` takes two arguments `a` and `b`, adds them together, and returns the result as a string.

**Correctness**

The function appears to be correct in terms of its intended behavior. However, it does not handle potential errors that may occur during the addition operation. For example, if the inputs are not numbers, the function will raise a TypeError.

**Type Safety**

The function does not check the types of its inputs, which can lead to unexpected behavior. It would be better to add type checking to ensure that the inputs are numbers.

**Maintainability**

The function is simple and easy to understand. However, it could benefit from some additional comments to explain its purpose an

In [14]:
#Small parameter experiments

#temperature
prompt = "Give me 3 new realistic, useful, new gen, and fire project ideas using local LLMs to catch with the trend in 2026."

print(ask(prompt, temperature=0.1, top_p=0.9, max_new_tokens=200))
print(ask(prompt, temperature=1.0, top_p=0.95, max_new_tokens=200))

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Here are three new, realistic, and useful project ideas that leverage local Large Language Models (LLMs) and cater to the trends of 2026:

**1. Personalized Mental Health Journaling Assistant**

Create a mobile app that utilizes a local LLM to generate personalized journal prompts, mood-tracking templates, and emotional intelligence-based insights. The app would:

* Use a local LLM to analyze user input (e.g., journal entries, search queries) and provide relevant, context-specific suggestions for improvement.
* Offer guided meditations, breathing exercises, and relaxation techniques tailored to the user's emotional state.
* Integrate with wearable devices or health trackers to monitor physical activity, sleep patterns, and other vital signs.

**2. AI-Powered Local Food Recommendation Platform**

Develop a web or mobile application that leverages a local LLM to provide users with personalized food recommendations based on their dietary needs, preferences, and location. The platform woul